# Семинар 4. От регуляризации к воспроизводимому ML-эксперименту

В лекции мы разобрали, **как ограничивать сложность модели и выбирать её настройки**. Здесь проверим эти идеи на данных и разберём инструменты, которые позволяют сравнивать модели без ошибок в подготовке признаков.

У нас две связанные истории. Сначала на данных об **аренде квартир** посмотрим, как меняются коэффициенты и ошибка после регуляризации, и собственноручно выполним кросс-валидацию. Затем решим задачу **классификации трёх видов пингвинов**: обработаем пропуски, приведём признаки к нужному масштабу, закодируем категории, соберём `Pipeline` и сравним kNN с многоклассовой логистической регрессией.

**Главный вопрос:** если качество изменилось, можем ли мы объяснить *что именно изменили* и *на каких данных это проверили*?

Все ячейки содержат готовый код. Их можно запускать по порядку и возвращаться к отдельным экспериментам позднее.

## Часть I. Регуляризация: что изменилось на данных об аренде

Используем синтетическую учебную таблицу квартир в условном городе. Целевая переменная `monthly_rent_thousand_rub` — месячная арендная плата в тысячах рублей. Признаки описывают площадь, число комнат, расстояние до центра и метро, возраст здания и другие свойства.

Здесь нас интересует не экономическая интерпретация каждого коэффициента, а **сравнение процедур обучения**: одна и та же линейная форма, но разные штрафы на коэффициенты.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import root_mean_squared_error

SEED = 42
DATA_DIR = Path("data")

### Читаем таблицу

Файлы данных лежат в подпапке `data/` рядом с ноутбуком. Сначала проверим размер и несколько строк, чтобы видеть реальные названия столбцов.

In [ ]:
rent = pd.read_csv(DATA_DIR / "apartment_rent_data.csv")
print("Размер:", rent.shape)
rent.head(4)

### Отделяем независимый test

Мы уже знакомы с этим правилом: итоговая проверка не участвует ни в выборе штрафа, ни в сравнении промежуточных моделей. Для первой демонстрации внутри dev сделаем ещё один train/validation split.

In [ ]:
X_rent = rent.drop(columns="monthly_rent_thousand_rub")
y_rent = rent["monthly_rent_thousand_rub"]

X_rent_dev, X_rent_test, y_rent_dev, y_rent_test = train_test_split(
    X_rent, y_rent, test_size=0.25, random_state=SEED
)
print("Dev:", X_rent_dev.shape, "Test:", X_rent_test.shape)

In [ ]:
X_rent_train, X_rent_valid, y_rent_train, y_rent_valid = train_test_split(
    X_rent_dev, y_rent_dev, test_size=0.25, random_state=11
)
print("Train:", len(X_rent_train), "Validation:", len(X_rent_valid))

### Сопоставим масштаб признаков

При одинаковом прогнозе коэффициенты меняются вместе с единицами измерения признаков. Но L1- и L2-штраф зависят от **самих коэффициентов**. Поэтому перед сравнением OLS, Ridge и Lasso стандартизируем признаки.

Пока выполним преобразование явно. Ниже мы подробно разберём, что именно обучается у `StandardScaler` при вызове `fit`.

In [ ]:
rent_scaler = StandardScaler()
X_rent_train_scaled = rent_scaler.fit_transform(X_rent_train)
X_rent_valid_scaled = rent_scaler.transform(X_rent_valid)
print("Форма данных:", X_rent_train_scaled.shape)

### Три модели с одной формой прогноза

Все три модели предсказывают $\hat y = \beta_0 + x^\top \beta$.

- **OLS** выбирает коэффициенты по минимальной сумме квадратов ошибок.
- **Ridge** добавляет L2-штраф; в sklearn его сила задаётся `alpha`.
- **Lasso** добавляет L1-штраф, поэтому отдельные коэффициенты могут оказаться точно нулевыми.

Значения `alpha=6` для Ridge и `alpha=0.5` для Lasso — *демонстрационные*, не выбранные по CV оптимумы. Их нельзя интерпретировать как одинаковую «силу» разных штрафов: в sklearn целевые функции Ridge и Lasso нормированы по-разному.

In [ ]:
ols_rent = LinearRegression().fit(X_rent_train_scaled, y_rent_train)

In [ ]:
ridge_rent = Ridge(alpha=6).fit(X_rent_train_scaled, y_rent_train)
lasso_rent = Lasso(alpha=0.5, max_iter=50_000).fit(
    X_rent_train_scaled, y_rent_train
)

### Что произошло с коэффициентами?

Сравним несколько связанных и нескольких других признаков. Площади квартиры часто коррелируют между собой, поэтому один и тот же прогноз можно получить при разном распределении вклада между ними.

In [ ]:
rent_coef = pd.DataFrame(
    {"OLS": ols_rent.coef_, "Ridge": ridge_rent.coef_, "Lasso": lasso_rent.coef_},
    index=X_rent_train.columns,
)
shown_rent_features = [
    "total_area_m2", "living_area_m2", "kitchen_area_m2", "rooms",
    "distance_center_km", "walk_to_metro_min",
]
rent_coef.loc[shown_rent_features].round(2)

Посмотрите прежде всего на `total_area_m2` и `living_area_m2`: Lasso может занулить одну площадь, оставив другую. Это **свойство выбранного на данной выборке решения**, а не доказательство отсутствия связи исключённого признака с арендой.

У Ridge ненулевые коэффициенты обычно сохраняются, но их величины перераспределяются и сдерживаются штрафом.

Коэффициенты относятся к стандартизированным признакам: увеличение признака на одно тренировочное стандартное отклонение меняет прогноз на соответствующий коэффициент при остальных фиксированных признаках. Это описание модели, а не причинный эффект.

### Сравниваем качество на двух частях

Если цель — прогноз на новых объектах, мало сравнить только train. Посмотрим RMSE на данных обучения и на отложенной validation-части. Единица RMSE та же, что у целевой переменной: тысячи рублей в месяц.

In [ ]:
rent_models = {"OLS": ols_rent, "Ridge": ridge_rent, "Lasso": lasso_rent}
rent_comparison_rows = []
for name, model in rent_models.items():
    rent_comparison_rows.append({
        "Модель": name,
        "Train RMSE": root_mean_squared_error(
            y_rent_train, model.predict(X_rent_train_scaled)
        ),
        "Validation RMSE": root_mean_squared_error(
            y_rent_valid, model.predict(X_rent_valid_scaled)
        ),
        "Ненулевых коэф.": int((np.abs(model.coef_) > 1e-8).sum()),
    })
rent_comparison = pd.DataFrame(rent_comparison_rows)
rent_comparison.round(2)

**На что обратить внимание.** OLS даёт меньшую train-RMSE, но Ridge и Lasso могут выиграть на validation. При этом Lasso использует меньше ненулевых коэффициентов. Одно удачное разбиение ещё не даёт надёжного основания выбрать гиперпараметры — оценка зависит от состава validation.

## Часть II. Кросс-валидация вручную

Возьмём одну фиксированную процедуру — **стандартизация → Ridge с `alpha=6`** — и оценим её на пяти разных разбиениях **внутри dev**.

Важно: каждый фолд — это *новое обучение* и стандартизатора, и Ridge. Нельзя один раз получить обученный scaler на всём dev, а затем делать вид, что validation-часть была независимой.

In [ ]:
rent_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
rent_folds = list(rent_cv.split(X_rent_dev))
train_idx, valid_idx = rent_folds[0]
print("Первый фолд — train:", len(train_idx), ", validation:", len(valid_idx))

### Один проход по шагам

Индексы из `split` — позиции строк внутри `X_rent_dev`, поэтому для выбора строк используем `iloc`. Обучим преобразование только по текущему train-фолду.

In [ ]:
X_rent_fold_train = X_rent_dev.iloc[train_idx]
X_rent_fold_valid = X_rent_dev.iloc[valid_idx]
y_rent_fold_train = y_rent_dev.iloc[train_idx]
y_rent_fold_valid = y_rent_dev.iloc[valid_idx]

In [ ]:
fold_scaler = StandardScaler()
X_fold_train_scaled = fold_scaler.fit_transform(X_rent_fold_train)
X_fold_valid_scaled = fold_scaler.transform(X_rent_fold_valid)

Теперь на преобразованных данных обучаем **новую** Ridge. Для validation выполняем `predict`, не повторяя `fit`.

In [ ]:
fold_ridge = Ridge(alpha=6).fit(X_fold_train_scaled, y_rent_fold_train)
fold_prediction = fold_ridge.predict(X_fold_valid_scaled)

In [ ]:
fold_first_rmse = root_mean_squared_error(y_rent_fold_valid, fold_prediction)
print(f"RMSE первого фолда: {fold_first_rmse:.3f} тыс. ₽")

### Повторяем пять проходов

В цикле нет «одной модели Ridge»: на каждой итерации создаются новые объекты `StandardScaler` и `Ridge`. Меняются как train-объекты, так и обученное состояние обоих шагов.

In [ ]:
rent_fold_rmse = []
for train_idx, valid_idx in rent_folds:
    X_tr, X_va = X_rent_dev.iloc[train_idx], X_rent_dev.iloc[valid_idx]
    y_tr, y_va = y_rent_dev.iloc[train_idx], y_rent_dev.iloc[valid_idx]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)

    model = Ridge(alpha=6).fit(X_tr_scaled, y_tr)
    error = root_mean_squared_error(y_va, model.predict(X_va_scaled))
    rent_fold_rmse.append(error)

In [ ]:
rent_cv_table = pd.DataFrame({
    "Фолд": range(1, 6), "Validation RMSE": rent_fold_rmse,
})
rent_cv_table.round(3)

In [ ]:
print("Средняя CV-RMSE:", round(np.mean(rent_fold_rmse), 3))
print("Корень из средней MSE:", round(np.sqrt(np.mean(np.square(rent_fold_rmse))), 3))

Мы усреднили **RMSE отдельных фолдов**. Это не совсем то же число, что $\sqrt{\text{средней MSE}}$: извлечение корня и усреднение не переставляются местами.

Важнее другое: одна настройка `alpha` получает *несколько оценок*, а не только результат одного случайного validation-разбиения. Позже научимся передавать **всю обучаемую процедуру** в `cross_val_score` и `GridSearchCV` вместо написания такого цикла вручную.

Обучающие части фолдов пересекаются, поэтому пять ошибок нельзя считать результатами пяти независимых экспериментов.

---

## Часть III. Три вида пингвинов: новая задача, тот же принцип честного сравнения

**Palmer Penguins** — реальные наблюдения за взрослыми пингвинами трёх видов на трёх островах архипелага Палмер у Антарктического полуострова. Таблица содержит **344 наблюдения и 8 столбцов**, собранных в 2007–2009 годах. По измерениям птицы и сведениям о наблюдении хотим определить **Adelie, Chinstrap или Gentoo**. Модель получает табличные признаки, а не фотографии.

Теперь ответ многоклассовый, а основной критерий — **macro-F1**. В отличие от RMSE, больший F1 означает лучшее качество. Усреднение *macro* даёт одинаковый вес каждому виду.

Данные [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/): Horst, Hill & Gorman (2020), первичные наблюдения — Kristen Gorman и Palmer Station LTER; исследование Gorman, Williams & Fraser (2014). Данные доступны по [CC0 1.0](https://creativecommons.org/publicdomain/zero/1.0/).

### Познакомимся с тремя видами

<img src="assets/lter_penguins.png" alt="Три вида пингвинов: слева Chinstrap, в центре Gentoo, справа Adelie — рисунок Allison Horst" width="760">

Слева направо: **Chinstrap**, **Gentoo**, **Adelie**. Иллюстрация помогает связать названия классов с птицами; изображения не используются как вход модели.

*Artwork by @allison_horst — Allison Horst. [Оригинал и разрешение на учебное использование](https://allisonhorst.github.io/palmerpenguins/articles/art.html). Рисунок без изменений.*

### Как измеряют клюв

<img src="assets/culmen_depth.png" alt="Длина клюва измеряется вдоль верхнего края, глубина (высота) — поперёк клюва; рисунок Allison Horst" width="650">

`bill_length_mm` — длина клюва, `bill_depth_mm` — его глубина (высота). На рисунке показаны два разных направления измерения: это самостоятельные числовые признаки, оба в миллиметрах.

*Artwork by @allison_horst — Allison Horst. [Оригинал и разрешение на учебное использование](https://allisonhorst.github.io/palmerpenguins/articles/art.html). Рисунок без изменений.*

### Что означают столбцы

| Столбец | Содержание | Тип |
|---|---|---|
| `species` | вид: Adelie, Chinstrap или Gentoo — наша целевая переменная | категория |
| `island` | остров наблюдения: Biscoe, Dream или Torgersen | номинальный признак |
| `bill_length_mm` | длина клюва, мм | число |
| `bill_depth_mm` | глубина (высота) клюва, мм | число |
| `flipper_length_mm` | длина ласта, мм | число |
| `body_mass_g` | масса тела, г | число |
| `sex` | пол птицы, если известен: female / male | номинальный признак с пропусками |
| `year` | год наблюдения: 2007–2009 (в этой постановке не используем) | целое число |

Для моделей оставим четыре числовых измерения и две категории. Год характеризует период сбора данных, но в наш набор признаков не входит.

Остров — настоящий признак, сильно связанный с видом. Сам по себе он не является утечкой: его допустимость зависит от того, знает ли его система **в момент классификации**. Здесь рассматриваем сценарий, где место наблюдения известно. Проверка относится к птицам из той же среды наблюдений, а не к переносу модели на ранее не исследованные острова.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, classification_report

penguins = pd.read_csv(DATA_DIR / "penguins.csv")
print("Размер:", penguins.shape)
penguins.head(5)

In [ ]:
penguins.info()

In [ ]:
penguins.isna().sum().to_frame("Пропусков")

Сразу видим две причины для подготовки данных: значения отдельных измерений отсутствуют, а названия острова и пола представлены **строками**, не координатами для kNN и не числовой матрицей для логистической регрессии.

### Фиксируем признаки и разделение

Возьмём четыре числовых измерения и две категории. Пока не добавляем `year`. Все последующие сравнения проводим на **одном и том же dev**; test откроем в самом конце.

In [ ]:
numeric_features = [
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"
]
categorical_features = ["island", "sex"]
feature_names = numeric_features + categorical_features
X_penguin = penguins[feature_names]
y_penguin = penguins["species"]

In [ ]:
X_penguin_dev, X_penguin_test, y_penguin_dev, y_penguin_test = train_test_split(
    X_penguin, y_penguin, test_size=0.25, random_state=SEED,
    stratify=y_penguin,
)
print("Dev:", X_penguin_dev.shape, "Test:", X_penguin_test.shape)

In [ ]:
y_penguin_dev.value_counts().rename("Число наблюдений").to_frame()

### Есть ли разделение между видами по измерениям?

Покажем два признака клюва. По ним уже можно заметить различия групп, но часть точек перекрывается. Этот график помогает понять, почему задача классификации содержательна и почему одному признаку не обязательно хватит информации для хорошего прогноза.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.4))
for species in ["Adelie", "Chinstrap", "Gentoo"]:
    mask = y_penguin_dev == species
    ax.scatter(
        X_penguin_dev.loc[mask, "bill_length_mm"],
        X_penguin_dev.loc[mask, "bill_depth_mm"],
        alpha=0.65, s=25, label=species,
    )
ax.set(xlabel="Длина клюва, мм", ylabel="Глубина клюва, мм")
ax.legend(title="Вид")
plt.show()

### Базовая точка отсчёта

`DummyClassifier(strategy="most_frequent")` всегда выбирает самый частый класс обучающей части. Признаки ему не нужны: его задача — показать, какое качество получается практически **без распознавания видов**.

`StratifiedKFold` старается поддерживать доли классов в каждом фолде. Для всех следующих моделей будем использовать те же пять разбиений и `scoring="f1_macro"`.

In [ ]:
penguin_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = "f1_macro"
penguin_dummy = DummyClassifier(strategy="most_frequent")
dummy_scores = cross_val_score(
    penguin_dummy, X_penguin_dev, y_penguin_dev, cv=penguin_cv, scoring=scoring
)
print("Macro-F1 по фолдам:", dummy_scores.round(3))
print("Средняя CV macro-F1:", round(dummy_scores.mean(), 3))

---

## Часть IV. Обучаемые преобразования: что остаётся после `fit`

Вспомним пропуски в измерениях. Простейшая идея — заменить неизвестную длину клюва **медианой** известных длин. Но медиану сначала нужно *вычислить на обучающих наблюдениях*.

Это означает, что даже такой, на первый взгляд, технический шаг **учится по данным**. Объект преобразования должен запомнить медиану, а затем использовать её при обработке новых строк.

In [ ]:
toy_bill = pd.DataFrame({"bill_length_mm": [37.1, np.nan, 41.0, 53.2]})
median_bill = toy_bill["bill_length_mm"].median()
toy_bill.assign(filled=toy_bill["bill_length_mm"].fillna(median_bill))

### `SimpleImputer`: `fit` и `transform`

`fit(X_train)` вычисляет и сохраняет нужную статистику (`statistics_`). `transform(X_new)` использует **сохранённое** значение, даже если распределение новых строк другое. В этом состоит смысл разделения двух операций.

`fit_transform(X_train)` последовательно выполняет обе операции на той же обучающей таблице. Для новых строк вызываем только `transform`, без повторного `fit`.

In [ ]:
from sklearn.impute import SimpleImputer

bill_imputer = SimpleImputer(strategy="median")
bill_imputer.fit(toy_bill)
print("Сохранённая медиана:", bill_imputer.statistics_)

In [ ]:
new_bills = pd.DataFrame({"bill_length_mm": [np.nan, 39.5]})
pd.DataFrame(
    bill_imputer.transform(new_bills), columns=["bill_length_mm"]
)

Если бы мы пересчитали медиану по `new_bills`, она стала бы другой. Но преобразователь **не переобучается** при получении новой птицы: его состояние получено по train.

In [ ]:
numeric_imputer_demo = SimpleImputer(strategy="median")
numeric_imputer_demo.fit(X_penguin_dev[numeric_features])
pd.Series(numeric_imputer_demo.statistics_, index=numeric_features, name="Медиана по dev")

В этой ячейке мы намеренно показываем состояние имьютера, обученного на всём dev, **только для демонстрации его устройства**. Этот экземпляр не будем передавать в CV: внутри CV медианы должны вычисляться заново на train-части каждого фолда.

### Масштабирование тоже обучается

`StandardScaler` вычитает среднее $\bar x_{\mathrm{train}}$ и делит на стандартное отклонение $s_{\mathrm{train}}$:

$$z=\frac{x-\bar x_{\mathrm{train}}}{s_{\mathrm{train}}}.$$

Для непостоянного признака `MinMaxScaler` использует тренировочные минимум и максимум:

$$x_{\mathrm{scaled}}=\frac{x-\min_{\mathrm{train}}}{\max_{\mathrm{train}}-\min_{\mathrm{train}}}.$$

Обучающий диапазон переводится в $[0,1]$, но новые значения могут выходить за его пределы. И тот, и другой преобразователь **запоминает параметры по данным**, а не просто применяет универсальную константу.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

mass_demo = pd.DataFrame({"mass": [3000, 4000, 5000, 6000]})
standard_demo = StandardScaler().fit(mass_demo)
minmax_demo = MinMaxScaler().fit(mass_demo)

In [ ]:
pd.DataFrame({
    "масса, г": mass_demo["mass"],
    "StandardScaler": standard_demo.transform(mass_demo).ravel(),
    "MinMaxScaler": minmax_demo.transform(mass_demo).ravel(),
}).round(3)

In [ ]:
print("Среднее:", standard_demo.mean_)
print("Масштаб:", standard_demo.scale_)
print("Минимум:", minmax_demo.data_min_)
print("Максимум:", minmax_demo.data_max_)

Итак, медиана, среднее, стандартное отклонение, минимум и максимум — **обученное состояние**. Следующая задача: не забывать повторять все эти действия при каждом новом обучении модели.

---

## Часть V. Первый `Pipeline`: последовательность для числовых признаков

Для kNN особенно важно, что единицы измерения меняют расстояния. Масса измеряется тысячами граммов, длина клюва — десятками миллиметров. Без подготовки масса легко начинает доминировать в евклидовом расстоянии.

Соберём цепочку: **заполнить пропуски → масштабировать → найти ближайших соседей**. У каждого шага будет имя, а последний шаг получит данные после всех преобразований.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

numeric_knn = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])
print("Шаги:", list(numeric_knn.named_steps))

### Что произойдёт при `numeric_knn.fit(X, y)`?

1. `SimpleImputer.fit_transform` узнает медианы по текущему train.
2. `StandardScaler.fit_transform` узнает средние и масштабы *после заполнения пропусков*.
3. `KNeighborsClassifier.fit` сохранит преобразованные обучающие наблюдения и их классы.

`predict` для новой строки последовательно вызывает `transform` у первых шагов и `predict` у kNN. На новой строке никто не пересчитывает медианы или средние.

In [ ]:
numeric_scores = cross_val_score(
    numeric_knn, X_penguin_dev[numeric_features], y_penguin_dev,
    cv=penguin_cv, scoring=scoring,
)
print("CV macro-F1 по фолдам:", numeric_scores.round(3))
print("Среднее:", round(numeric_scores.mean(), 3))

### Изолируем эффект scaling

Сохраним `k=5`, одинаковые признаки, `SimpleImputer` и разбиения CV. **Изменим только правило масштабирования.** Это проверяемая гипотеза об изменении представления данных, а не сравнение разных алгоритмов.

In [ ]:
knn_no_scaling = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])
knn_minmax = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])

In [ ]:
scores_no_scaling = cross_val_score(
    knn_no_scaling, X_penguin_dev[numeric_features], y_penguin_dev,
    cv=penguin_cv, scoring=scoring,
)
scores_minmax = cross_val_score(
    knn_minmax, X_penguin_dev[numeric_features], y_penguin_dev,
    cv=penguin_cv, scoring=scoring,
)

In [ ]:
scaling_comparison = pd.DataFrame({
    "Без scaling": scores_no_scaling,
    "StandardScaler": numeric_scores,
    "MinMaxScaler": scores_minmax,
}, index=[f"Фолд {k}" for k in range(1, 6)])
scaling_comparison.round(3)

In [ ]:
scaling_comparison.mean().rename("Средняя CV macro-F1").round(3).to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3))
scaling_comparison.mean().sort_values().plot.barh(ax=ax)
ax.set(xlabel="Средняя CV macro-F1", ylabel="", xlim=(0, 1))
ax.set_title("Один kNN, разная подготовка числовых признаков")
plt.tight_layout()
plt.show()

**Результат.** Здесь средняя CV macro-F1 выросла с **0,653 без scaling** до **0,967** с обоими вариантами масштабирования. Сами классы и правило голосования не менялись; изменилась **геометрия расстояний**. Это не универсальное утверждение, что любой алгоритм всегда выигрывает от стандартизации.

---

## Часть VI. Категориальные признаки: от проблемы числовых меток до one-hot

До сих пор модель использовала только четыре числовых измерения. Но у нас есть ещё `island` и `sex`. Для них значения `Dream`, `Biscoe`, `Torgersen`, `female`, `male` — **названия категорий, а не измерения**.

Как превратить их в числа, не придумав случайно несуществующие отношения между категориями?

In [ ]:
X_penguin_dev[categorical_features].head(6)

In [ ]:
pd.crosstab(y_penguin_dev, X_penguin_dev["island"])

### Почему коды `0, 1, 2` — плохой выбор

Допустим, мы назначили `Biscoe=0`, `Dream=1`, `Torgersen=2`. В kNN это означает, что расстояние между Biscoe и Dream равно 1, а между Biscoe и Torgersen — 2. Но такие расстояния не следуют из смысла названий островов.

Для логистической регрессии проблема тоже есть: один коэффициент при коде острова вынуждает модель считать переходы `0→1` и `1→2` одинаковыми шагами по линейному score. Любая перестановка кодов могла бы изменить результат.

### One-hot: отдельный индикатор каждой категории

Вместо одного произвольно пронумерованного столбца создаём несколько признаков:

| Остров | `island_Biscoe` | `island_Dream` | `island_Torgersen` |
|---|---:|---:|---:|
| Biscoe | 1 | 0 | 0 |
| Dream | 0 | 1 | 0 |
| Torgersen | 0 | 0 | 1 |

Теперь разные известные острова представлены разными единичными векторами. Их попарные евклидовы расстояния одинаковы: $\sqrt{(1-0)^2+(0-1)^2}=\sqrt{2}$. Мы убрали **ложный числовой порядок**, но не получили объективную географическую меру расстояния между островами.

### Почему недостаточно разового `pd.get_dummies`?

При обучении и применении модели число признаков и порядок их столбцов должны совпадать. Если в новой партии данных нет одной категории или появилась незнакомая, разовое кодирование может создать **другой набор колонок**.

`OneHotEncoder` — обучаемый преобразователь: `fit` запоминает найденные категории и их порядок, а `transform` придерживается этого же пространства признаков.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

island_toy = pd.DataFrame({"island": ["Biscoe", "Dream", "Torgersen"]})
island_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
island_encoder.fit(island_toy)
print("Запомнил категории:", island_encoder.categories_)

In [ ]:
island_code = island_encoder.transform(island_toy)
pd.DataFrame(
    island_code, columns=island_encoder.get_feature_names_out(),
    index=island_toy["island"],
)

### А если встретился новый остров?

По умолчанию кодировщик может выдать ошибку, если в `transform` появляется категория, отсутствовавшая при `fit`. В учебном примере выбираем `handle_unknown="ignore"`: незнакомый остров получает **нули во всех индикаторах этого столбца**.

Это не значит, что модель узнала что-то о новом острове. Она лишь умеет обработать неизвестное название без падения. При нескольких категориальных столбцах остальные признаки продолжают кодироваться как обычно.

In [ ]:
unknown_island = pd.DataFrame({"island": ["Unknown island"]})
pd.DataFrame(
    island_encoder.transform(unknown_island),
    columns=island_encoder.get_feature_names_out(),
)

### Куда делись пропуски в `sex`?

До one-hot можно заполнить пропущенную категорию наиболее частой **на текущем обучающем фолде**. Это ещё одно обучаемое преобразование.

Собираем **категориальную ветвь**: `SimpleImputer(most_frequent) → OneHotEncoder`. Порядок важен: кодировщик получает уже заполненные категории.

In [ ]:
categorical_branch = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
print("Шаги категориальной ветви:", list(categorical_branch.named_steps))

In [ ]:
categorical_demo = categorical_branch.fit(X_penguin_dev[categorical_features])
print("Пропущенный sex заполняем:", categorical_demo.named_steps["imputer"].statistics_[1])
print("Запомненные категории:", categorical_demo.named_steps["onehot"].categories_)

In [ ]:
cat_sample = X_penguin_dev[categorical_features].head(5)
cat_names = categorical_demo.named_steps["onehot"].get_feature_names_out(categorical_features)
pd.DataFrame(
    categorical_demo.transform(cat_sample), columns=cat_names, index=cat_sample.index
)

Здесь снова встретились **обучаемые состояния**: `SimpleImputer.statistics_` и `OneHotEncoder.categories_`. Для новых наблюдений обе операции применяются в том же порядке. One-hot делает категории пригодными для стандартных числовых моделей, но выбор кодировки и её влияние на расстояния и регуляризацию остаются содержательными решениями.

В нашей схеме one-hot индикаторы не проходят через StandardScaler: числовые измерения и категориальные индикаторы готовятся в разных ветвях.

---

## Часть VII. `ColumnTransformer`: разные столбцы — разные цепочки

Теперь понятна следующая проблема. Измерения тела требуют числовой импутации и масштабирования; названия островов и пола — категориальной импутации и one-hot.

Один последовательный `Pipeline` применяет операции ко *всей полученной таблице*. А нам нужны **параллельные ветви**, выбирающие свои столбцы, с последующим объединением результатов:

- числа → медиана → StandardScaler;
- категории → самая частая категория → OneHotEncoder;
- объединённая числовая матрица → классификатор.

In [ ]:
numeric_branch = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
print("Числовая ветвь:", list(numeric_branch.named_steps))

### Как описывается одна ветвь `ColumnTransformer`

Каждый элемент списка имеет три части: `("имя", преобразователь, список_столбцов)`. Имя нужно для обращения к ветви, преобразователь задаёт последовательность действий, а список указывает, к каким исходным столбцам она применяется.

Результаты объединяются в порядке перечисления ветвей. Неуказанные столбцы по умолчанию отбрасываются (`remainder="drop"`).

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", numeric_branch, numeric_features),
    ("cat", categorical_branch, categorical_features),
])
print("Ветви:", [name for name, _, _ in preprocessor.transformers])

### Посмотрим на результат *для понимания устройства*

Выполним `fit_transform` **отдельного демонстрационного экземпляра** на всём dev, чтобы увидеть размеры и названия получившихся столбцов. Для настоящей CV это состояние использовать **не будем**: каждая копия `preprocessor` должна обучаться только на train-фолде.

`clone(preprocessor)` создаёт объект с той же схемой и гиперпараметрами, но **без обученного состояния**. Поэтому демонстрационное обучение не меняет исходную спецификацию.

In [ ]:
from sklearn.base import clone

preprocessor_demo = clone(preprocessor)
X_penguin_ready = preprocessor_demo.fit_transform(X_penguin_dev)
print("Исходная таблица:", X_penguin_dev.shape)
print("После преобразования:", X_penguin_ready.shape)

In [ ]:
ready_columns = preprocessor_demo.get_feature_names_out()
print("Новые имена:", list(ready_columns))

In [ ]:
pd.DataFrame(
    X_penguin_ready, columns=ready_columns, index=X_penguin_dev.index
).head(5).round(2)

Четыре числовых измерения остаются четырьмя колонками. `island` превращается в три индикатора, `sex` — в два. Поэтому вместо 6 исходных столбцов получаем **9 числовых признаков**.

Преобразование расширило представление, но *ни один вид пингвина ещё не был предсказан*: пока мы готовили только матрицу признаков.

### Теперь добавим модель

Внешний `Pipeline` получает **обычный исходный DataFrame**. Его первый шаг `preprocess` обучает обе ветви `ColumnTransformer`, а второй `model` обучает классификатор на подготовленных числах. Когда позже вызовем `predict`, он повторит все сохранённые преобразования без нового `fit`.

In [ ]:
full_knn = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", KNeighborsClassifier(n_neighbors=5)),
])
print("Внешние шаги:", list(full_knn.named_steps))

---

## Часть VIII. Почему CV получает **весь Pipeline**

Вручную мы уже проводили CV для квартир: для каждого фолда заново обучали и scaler, и Ridge. Теперь то же правило распространяется сразу на **медианы, масштаб, категории и классификатор**.

Посмотрим на одном фолде, чем отличается среднее, оценённое только по train, от среднего по всему dev. Валидирующие строки не должны влиять на состояние преобразования.

In [ ]:
penguin_train_idx, penguin_valid_idx = next(
    penguin_cv.split(X_penguin_dev, y_penguin_dev)
)
print("Train:", len(penguin_train_idx), "Validation:", len(penguin_valid_idx))

In [ ]:
X_penguin_fold_train = X_penguin_dev.iloc[penguin_train_idx]
proper_fold_prep = clone(numeric_branch).fit(X_penguin_fold_train[numeric_features])
leaky_dev_prep = clone(numeric_branch).fit(X_penguin_dev[numeric_features])

In [ ]:
pd.DataFrame({
    "Только train-фолд": proper_fold_prep.named_steps["scaler"].mean_,
    "Весь dev (ошибка для CV)": leaky_dev_prep.named_steps["scaler"].mean_,
}, index=numeric_features).round(3)

Даже если различия средних небольшие, методологическая ошибка реальна: в «плохой» версии validation уже участвовал в `fit`. В другом наборе это могло бы существенно изменить результат. Нельзя обещать, что любая утечка обязательно повысит метрику.

Правильная схема — передавать CV **целую необученную цепочку**, чтобы библиотека заново обучала её на каждом train-фолде.

### Автоматизируем полную процедуру

`cross_val_score(full_knn, ...)` на каждом проходе создаёт независимую копию Pipeline, заново обучает все шаги и оценивает прогнозы на соответствующем validation-фолде. Test в этой процедуре не участвует.

In [ ]:
full_knn_scores = cross_val_score(
    full_knn, X_penguin_dev, y_penguin_dev,
    cv=penguin_cv, scoring=scoring,
)
print("Фолды:", full_knn_scores.round(3))
print("Средняя CV macro-F1:", round(full_knn_scores.mean(), 3))

Мы уже увидели, как оценивается **одна заданная цепочка**. Теперь хотим выбрать её настройки. Поэтому повторим полную CV для нескольких наборов параметров и сравним средние оценки.

---

## Часть IX. Выбор гиперпараметров и сравнение моделей

В регрессии мы меняли регуляризацию коэффициентов. У kNN регулируем число соседей `k`, способ голосования и метрику расстояния. У логистической регрессии меняем `C`, обратную характеристику силы регуляризации: **меньше `C` — сильнее штраф**.

`GridSearchCV` автоматизирует процедуру выбора. Важно, что `estimator` для него — **наш полный Pipeline**, а не только последний классификатор.

### Как адресовать параметр внутри Pipeline?

Внешняя цепочка содержит шаг `model`. Чтобы передать значение параметра этому шагу, используем синтаксис `model__имя_параметра`: два подчёркивания отделяют имя шага от имени его настройки.

Например, `model__n_neighbors` относится к kNN, а `model__C` — к логистической регрессии. Сам `preprocess` остаётся в составе каждой оцениваемой процедуры.

In [ ]:
from sklearn.model_selection import GridSearchCV

knn_grid = {
    "model__n_neighbors": [1, 2, 3, 4, 5, 7, 9, 11, 15, 21],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
}
print("Комбинаций kNN:", 10 * 2 * 2)

In [ ]:
knn_search = GridSearchCV(
    full_knn, knn_grid, cv=penguin_cv,
    scoring=scoring, refit=True, n_jobs=1,
)

При 40 конфигурациях и пяти фолдах будет **200 обучений** для оценки качества, а затем ещё одно (`refit=True`) на **всём dev** для выбранной конфигурации. Здесь каждое обучение включает все преобразователи, а не только поиск соседей.

In [ ]:
knn_search.fit(X_penguin_dev, y_penguin_dev)
print("Поиск kNN завершён.")

In [ ]:
print("Лучшие настройки:", knn_search.best_params_)
print("CV macro-F1:", round(knn_search.best_score_, 4))
print("Обученный k:", knn_search.best_estimator_.named_steps["model"].n_neighbors)

### Что находится в `cv_results_`?

Каждая строка таблицы соответствует проверенной комбинации. Некоторые настройки дают одинаковый средний F1 — например, потому что на небольших validation-частях несколько вариантов предсказывают одинаковые классы.

Одинаковый балл не означает, что обученные модели тождественны или что разница в четвёртом знаке имеет большое содержательное значение.

`mean_test_score` здесь означает среднюю оценку **validation-фолдов CV**. Слово `test` в имени атрибута sklearn не относится к нашему отложенному `X_penguin_test`.

In [ ]:
knn_results = pd.DataFrame(knn_search.cv_results_)
knn_results[[
    "param_model__n_neighbors", "param_model__weights",
    "param_model__metric", "mean_test_score", "rank_test_score",
]].sort_values("rank_test_score").head(10).round(3)

### Меняем алгоритм, сохраняем эксперимент

Многоклассовая логистическая регрессия строит совместные class scores и вероятности через softmax. kNN выбирает классы по соседям. Это разные механизмы, но теперь мы можем сравнить их **на одинаковых признаках, одинаковой обработке, одинаковых CV-фолдах и одной метрике**.

`clone(preprocessor)` создаёт новую копию спецификации преобразований **без обученных статистик**, чтобы не переносить состояние между моделями.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=3000)),
])
print("Шаги:", list(logreg_pipeline.named_steps))

In [ ]:
logreg_grid = {"model__C": [0.01, 0.1, 1, 10, 100]}
logreg_search = GridSearchCV(
    logreg_pipeline, logreg_grid, cv=penguin_cv,
    scoring=scoring, refit=True, n_jobs=1,
)

In [ ]:
logreg_search.fit(X_penguin_dev, y_penguin_dev)
print("Поиск LogisticRegression завершён.")

In [ ]:
print("Выбранный C:", logreg_search.best_params_)
print("CV macro-F1:", round(logreg_search.best_score_, 4))

In [ ]:
logreg_results = pd.DataFrame(logreg_search.cv_results_)
logreg_results[["param_model__C", "mean_test_score", "std_test_score"]].round(3)

**Вернулись к регуляризации.** Если `C` очень маленькое, штраф может слишком ограничить логистическую регрессию. Подбор по CV показывает, какое значение лучше работает в данной постановке. Близкие оценки при `C=1` и `C=10` не стоит интерпретировать как принципиальную разницу между моделями.

### Все кандидаты в одной таблице

Baseline, kNN и LogisticRegression проверялись по одинаковой схеме CV на одном dev. У моделей разные гиперпараметры, но сравнение качества происходит **одним и тем же способом**.

In [ ]:
penguin_comparison = pd.DataFrame({
    "Модель": ["Dummy: самый частый класс", "kNN (настроен)", "LogisticRegression (настроена)"],
    "CV macro-F1": [dummy_scores.mean(), knn_search.best_score_, logreg_search.best_score_],
})
penguin_comparison.round(3)

Что означают три атрибута поиска?

- `best_params_` — выбранные значения гиперпараметров.
- `best_score_` — лучшая **средняя CV-оценка** среди проверенных конфигураций; после подбора она может быть несколько оптимистичной.
- `best_estimator_` — **уже обученный заново на всём dev** Pipeline при `refit=True`, готовый принять исходный DataFrame.

Ни один из этих атрибутов не является оценкой на независимом test.

---

## Часть X. Где ошибаются три класса?

Общая macro-F1 может скрывать разный характер ошибок для Adelie, Chinstrap и Gentoo. Нам нужны прогнозы для **конкретных dev-объектов**, полученные без обучения на этих же объектах.

### Out-of-fold прогнозы (OOF)

В каждом фолде обучаем модель на четырёх частях и предсказываем классы оставшейся. Складывая результаты, получаем **один отложенный прогноз для каждого объекта dev**. Именно это делает `cross_val_predict`.

Важная оговорка: гиперпараметры уже выбирались по всему dev. Поэтому OOF-прогнозы с выбранными настройками удобны **для изучения ошибок**, но не становятся новой независимой оценкой качества всей процедуры подбора. Для неё остаётся test.

In [ ]:
class_order = ["Adelie", "Chinstrap", "Gentoo"]
knn_oof = cross_val_predict(
    knn_search.best_estimator_, X_penguin_dev, y_penguin_dev, cv=penguin_cv
)
logreg_oof = cross_val_predict(
    logreg_search.best_estimator_, X_penguin_dev, y_penguin_dev, cv=penguin_cv
)

### Матрицы ошибок

Строки ниже — **истинный вид**, столбцы — **предсказанный**. Значения вне диагонали указывают, какие виды классификатор путает. Для сравнения используем одинаковый порядок классов.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, pred, name in zip(
    axes, [knn_oof, logreg_oof], ["kNN", "LogisticRegression"]
):
    ConfusionMatrixDisplay.from_predictions(
        y_penguin_dev, pred, labels=class_order,
        ax=ax, cmap="Blues", colorbar=False,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

### Метрики по классам

`support` показывает, сколько настоящих объектов соответствующего вида было в dev. `recall` — долю распознанных объектов данного вида, `precision` — точность среди объектов, отнесённых моделью к этому виду. `macro avg` усредняет классовые показатели поровну.

In [ ]:
print(classification_report(
    y_penguin_dev, knn_oof, labels=class_order,
    digits=3, zero_division=0,
))

In [ ]:
print("LogisticRegression: macro-F1 OOF =", round(
    f1_score(y_penguin_dev, logreg_oof, average="macro"), 3
))
print(classification_report(
    y_penguin_dev, logreg_oof, labels=class_order,
    digits=3, zero_division=0,
))

### Какие именно объекты были ошибочными?

Сохраним рядом исходные измерения, истинный вид и обе предсказанные метки. Так можно увидеть, совпадают ли ошибки моделей и встречаются ли необычные строки с пропущенными измерениями.

In [ ]:
penguin_oof_detail = X_penguin_dev.copy()
penguin_oof_detail["Истинный вид"] = y_penguin_dev
penguin_oof_detail["kNN"] = knn_oof
penguin_oof_detail["LogReg"] = logreg_oof
penguin_errors = penguin_oof_detail[
    (penguin_oof_detail["Истинный вид"] != penguin_oof_detail["kNN"])
    | (penguin_oof_detail["Истинный вид"] != penguin_oof_detail["LogReg"])
]
print("Объектов с ошибкой хотя бы одной модели:", len(penguin_errors))
penguin_errors

По этим наблюдениям модели почти идеальны. Здесь полезнее освоить процедуру получения ошибок по классам, чем приписывать глубокий смысл разнице в одном-двух объектах. В более сложной задаче из ДЗ №2 содержательная диагностика будет гораздо важнее.

---

## Часть XI. Независимый test и новая исходная строка

Мы завершили сравнение на dev: выбрали две процедуры и их настройки. Теперь **однократно** проверим их на отложенном test. По результату этой проверки не будем заново выбирать `C`, `k` или набор признаков.

In [ ]:
best_knn = knn_search.best_estimator_
best_logreg = logreg_search.best_estimator_

penguin_test_pred_knn = best_knn.predict(X_penguin_test)
penguin_test_pred_logreg = best_logreg.predict(X_penguin_test)

In [ ]:
test_comparison = pd.DataFrame({
    "Модель": ["kNN", "LogisticRegression"],
    "Test accuracy": [
        accuracy_score(y_penguin_test, penguin_test_pred_knn),
        accuracy_score(y_penguin_test, penguin_test_pred_logreg),
    ],
    "Test macro-F1": [
        f1_score(y_penguin_test, penguin_test_pred_knn, average="macro"),
        f1_score(y_penguin_test, penguin_test_pred_logreg, average="macro"),
    ],
})
test_comparison.round(3)

### Вся обработка уже внутри обученной модели

`best_logreg` — это не «голый» классификатор. В его `preprocess` сохранены медианы, средние, масштабы и категории, найденные на **всём dev** после refit.

Поэтому новым объектам достаточно предоставить шесть исходных признаков в тех же столбцах. Ручной `fillna`, scaling или one-hot перед `predict` не нужны.

In [ ]:
new_penguins = pd.DataFrame({
    "bill_length_mm": [39.4, 49.6, 47.5],
    "bill_depth_mm": [18.5, 15.6, 19.3],
    "flipper_length_mm": [189.0, 218.0, 198.0],
    "body_mass_g": [3800.0, 5200.0, 3950.0],
    "island": ["Torgersen", "Biscoe", "Dream"],
    "sex": ["male", "female", np.nan],
})
new_penguins

In [ ]:
new_probabilities = best_logreg.predict_proba(new_penguins)
new_labels = best_logreg.predict(new_penguins)
print("Предсказанные виды:", list(new_labels))
print("Сумма вероятностей по строкам:", new_probabilities.sum(axis=1).round(5))

In [ ]:
pd.DataFrame(
    new_probabilities,
    columns=[f"P({name})" for name in best_logreg.classes_],
    index=["Пингвин 1", "Пингвин 2", "Пингвин 3"],
).round(3)

### Главное из двух экспериментов

- Регуляризация меняет процедуру выбора коэффициентов; снижение train-ошибки само по себе не означает улучшение новых прогнозов.
- CV оценивает повторно обученную **процедуру**, включая подготовку данных, а не один уже обученный объект.
- `fit` сохраняет статистики и категории; `transform` использует их без подглядывания в новые данные.
- Для разных типов столбцов нужны разные ветви `ColumnTransformer`, а целый `Pipeline` делает их частью проверяемой модели.
- Модели и их настройки сравниваются на одних фолдах и по одной метрике; multiclass-диагностика показывает ошибки за средним score.

**В ДЗ №2** предстоит пройти такой же путь самостоятельно для четырёх классов длительности поездки: выбрать допустимые признаки, сформулировать гипотезы FE, сравнить модели по macro-F1 и подготовить прогнозы для закрытого теста.

---

# Дополнительные эксперименты

Эти законченные эксперименты продолжают основной пример. Их удобно запускать самостоятельно после занятия: они не нужны для работы основных ячеек, а все важные переменные уже определены выше.

## Дополнение 1. Выбор Ridge `alpha` с помощью `GridSearchCV`

В основном маршруте для квартир мы оценили фиксированную `Ridge(alpha=6)` вручную. Теперь передадим библиотеке целую цепочку `scale → ridge` и переберём несколько значений `alpha`. Библиотека считает **отрицательную RMSE**, поскольку у неё больший `score` означает лучшее качество.

In [ ]:
rent_ridge_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", Ridge()),
])
rent_alpha_grid = {"ridge__alpha": [0.01, 0.1, 0.3, 1, 3, 6, 10, 30, 100, 300]}

In [ ]:
rent_ridge_search = GridSearchCV(
    rent_ridge_pipeline, rent_alpha_grid,
    cv=rent_cv, scoring="neg_root_mean_squared_error",
    return_train_score=True, refit=True,
)
rent_ridge_search.fit(X_rent_dev, y_rent_dev)
print("Поиск Ridge завершён.")

In [ ]:
print("Лучший alpha:", rent_ridge_search.best_params_)
print("CV RMSE:", round(-rent_ridge_search.best_score_, 3))
print("Итоговый шаг:", rent_ridge_search.best_estimator_.named_steps["ridge"])

In [ ]:
rent_ridge_results = pd.DataFrame(rent_ridge_search.cv_results_)
pd.DataFrame({
    "alpha": rent_ridge_results["param_ridge__alpha"].astype(float),
    "Train RMSE": -rent_ridge_results["mean_train_score"],
    "CV RMSE": -rent_ridge_results["mean_test_score"],
}).round(3)

Средняя train-RMSE и средняя validation-RMSE могут вести себя по-разному при усилении штрафа. Выбранное значение `alpha` минимизирует CV-RMSE **среди проверенных кандидатов**, а не среди всех математически возможных значений.

## Дополнение 2. Исключение острова — другая постановка

Место наблюдения очень информативно, но не в каждом сценарии будет известно. Проверим задачу, где модель видит **только собственные измерения и пол**. Повторим настройку обеих моделей по тем же фолдам и метрике.

Это **не автоматическое исправление утечки**: мы сознательно рассматриваем другую информацию на входе.

In [ ]:
numeric_no_island = numeric_features
categorical_no_island = ["sex"]
X_dev_no_island = X_penguin_dev[numeric_no_island + categorical_no_island]

In [ ]:
preprocessor_no_island = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_no_island),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_no_island),
])

In [ ]:
knn_no_island = Pipeline([
    ("preprocess", clone(preprocessor_no_island)),
    ("model", KNeighborsClassifier()),
])
knn_no_island_search = GridSearchCV(
    knn_no_island, knn_grid, cv=penguin_cv,
    scoring=scoring, n_jobs=1,
)

In [ ]:
knn_no_island_search.fit(X_dev_no_island, y_penguin_dev)
print("kNN без island — CV macro-F1:", round(knn_no_island_search.best_score_, 4))

In [ ]:
logreg_no_island = Pipeline([
    ("preprocess", clone(preprocessor_no_island)),
    ("model", LogisticRegression(max_iter=3000)),
])
logreg_no_island_search = GridSearchCV(
    logreg_no_island, logreg_grid, cv=penguin_cv,
    scoring=scoring, n_jobs=1,
)

In [ ]:
logreg_no_island_search.fit(X_dev_no_island, y_penguin_dev)
print("LogReg без island — CV macro-F1:", round(logreg_no_island_search.best_score_, 4))

In [ ]:
pd.DataFrame({
    "Модель": ["kNN", "LogisticRegression"],
    "С island": [knn_search.best_score_, logreg_search.best_score_],
    "Без island": [knn_no_island_search.best_score_, logreg_no_island_search.best_score_],
}).round(4)

Разница качества между двумя постановками небольшая, но видна. Она не измеряет «чистую причинную пользу острова»: это сравнение предсказательных процедур с разным набором входных признаков. Различия небольшого размера на таком датасете не стоит переинтерпретировать.

## Дополнение 3. Что находится внутри обученного Pipeline и как его сохранить

После `refit` модель хранит обученное состояние. Внутри `preprocess` — две ветви, а внутри каждой — свои параметры. Это можно проверить без нового вызова `.fit`.

In [ ]:
fitted_prep = best_logreg.named_steps["preprocess"]
fitted_num = fitted_prep.named_transformers_["num"]
fitted_cat = fitted_prep.named_transformers_["cat"]
print("Медианы:", fitted_num.named_steps["imputer"].statistics_.round(2))
print("Средние:", fitted_num.named_steps["scaler"].mean_.round(2))
print("Категории:", fitted_cat.named_steps["onehot"].categories_)

У многоклассовой логистической регрессии `coef_` содержит одну строку коэффициентов для каждого класса, а столбцы соответствуют **признакам после preprocessing**. Это наглядная связь практического API с softmax-моделью из лекции.

In [ ]:
fitted_clf = best_logreg.named_steps["model"]
coefficient_names = fitted_prep.get_feature_names_out()
pd.DataFrame(
    fitted_clf.coef_, index=fitted_clf.classes_, columns=coefficient_names,
).round(2)

Обученную цепочку можно сохранить целиком: иначе при новом запуске придётся восстанавливать не только коэффициенты классификатора, но и все медианы, масштабы и названия категорий.

In [ ]:
from joblib import dump, load

model_dir = Path("artifacts")
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "penguins_logreg_full_pipeline.joblib"
dump(best_logreg, model_path)
print("Файл сохранён:", model_path)

In [ ]:
loaded_logreg = load(model_path)
print("Вероятности совпадают:", np.allclose(
    loaded_logreg.predict_proba(new_penguins),
    best_logreg.predict_proba(new_penguins),
))

Загружайте `joblib` и другие pickle-подобные файлы **только из доверенных источников**: при загрузке может выполняться произвольный Python-код. Для переносимости важно также фиксировать версии Python и библиотек; сериализованные sklearn-объекты не гарантированно совместимы со всеми версиями.

## Дополнение 4. Как меняются конкретные соседи после scaling

Высокий прирост macro-F1 после стандартизации имеет геометрическое объяснение: порядок ближайших соседей может измениться. Посмотрим на один validation-объект и его пять ближайших train-строк в двух пространствах.

In [ ]:
train_ids, valid_ids = next(penguin_cv.split(X_penguin_dev, y_penguin_dev))
X_neighbor_train = X_penguin_dev.iloc[train_ids][numeric_features]
X_neighbor_valid = X_penguin_dev.iloc[valid_ids][numeric_features]

In [ ]:
from sklearn.metrics import pairwise_distances

raw_imputer = SimpleImputer(strategy="median")
X_train_raw = raw_imputer.fit_transform(X_neighbor_train)
X_valid_raw = raw_imputer.transform(X_neighbor_valid)
raw_distances = pairwise_distances(X_valid_raw[:1], X_train_raw)[0]
print("Пять ближайших без scaling:", np.argsort(raw_distances)[:5].tolist())

In [ ]:
neighbor_scaler = StandardScaler()
X_train_std = neighbor_scaler.fit_transform(X_train_raw)
X_valid_std = neighbor_scaler.transform(X_valid_raw)
std_distances = pairwise_distances(X_valid_std[:1], X_train_std)[0]
print("Пять ближайших после scaling:", np.argsort(std_distances)[:5].tolist())

Теперь можно объяснить прирост качества содержательно: изменились не параметры классификатора, а само **окружение объекта в пространстве признаков**. Поэтому настройка и оценивание kNN невозможны в отрыве от правил подготовки данных.